In [ ]:
import os

for variable in (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
):
    os.environ.setdefault(variable, "1")

from pathlib import Path
from datetime import datetime
import itertools
import json
import time
import numpy as np
import mindquantum as mq
from mindquantum.core.circuit import Circuit
from mindquantum.core.gates import RY, RZ, Z as MQZ, PauliChannel
from mindquantum.core.operators import QubitOperator, Hamiltonian
from mindquantum.simulator import Simulator

NQ = 4
NINPUT = NPARAM = 2 * NQ
DIM = 2**NQ
NPAULI = 4**NQ
NGRID = 3**NINPUT
NSECTOR = NINPUT + 1
GAMMAS = np.array([0.0, 0.05, 0.15, 0.30])
SEEDS = tuple(range(42, 62))
PILOT_SEEDS = (1001, 1002, 1003)
HALF_STEP_SEEDS = SEEDS[:5]
STEP_CANDIDATES = (0.10, 0.05, 0.025, 0.0125)
PILOT_TIME = 20.0
TRAINING_TIME = 200.0
SAVE_RESULTS = True
OUTPUT_ROOT = Path("results_notebook")

CONDITIONS = [
    {"task": "common", "gamma_index": i, "gamma": float(gamma)}
    for i, gamma in enumerate(GAMMAS)
] + [
    {"task": "matched", "gamma_index": i, "gamma": float(GAMMAS[i])} for i in (1, 2, 3)
]
print(f"NumPy {np.__version__}; MindQuantum {mq.__version__}")
print(
    f"{len(CONDITIONS)} conditions; {len(SEEDS)} paired initializations; {NGRID} grid points"
)


In [ ]:
I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.diag([1, -1]).astype(complex)


def embed(op, q):
    a = np.ones((1, 1), dtype=complex)
    for site in reversed(range(NQ)):
        a = np.kron(a, op if site == q else I)
    return a


def cz(a, b):
    return np.diag([-1 if s >> a & 1 and s >> b & 1 else 1 for s in range(DIM)]).astype(
        float
    )


C = np.eye(DIM)
for q in range(NQ - 1):
    C = cz(q, q + 1) @ C
K = np.stack([(-1j * embed(Y, q)).real for q in range(NQ)])
EYE = np.eye(DIM)
S = 34 + 13 * np.sqrt(2)
OBSERVABLE_TERMS = [
    ("Z0", np.sqrt(2) / S),
    ("Z0 Z1", 2 / S),
    ("Z0 Z1 Z2", 2 / S),
    ("X0", 2 / S),
    ("Z0 Z1 Z2 Z3", 4 / S),
    ("X0 X1", 4 * np.sqrt(2) / S),
    ("X0 X3", 8 / S),
    ("X0 X1 X2", -8 * np.sqrt(2) / S),
    ("X0 X1 X2 X3", 16 / S),
]
O = np.zeros((DIM, DIM), complex)
for label, w in OBSERVABLE_TERMS:
    op = np.eye(DIM, dtype=complex)
    for word in label.split():
        op = op @ embed(dict(X=X, Y=Y, Z=Z)[word[0]], int(word[1:]))
    O += w * op
assert abs(O.imag).max() == 0
O = O.real
PAULI_LABELS = []
paulis = []
for labels in itertools.product("IXYZ", repeat=NQ):
    op = np.ones((1, 1), dtype=complex)
    for q in reversed(range(NQ)):
        op = np.kron(op, dict(I=I, X=X, Y=Y, Z=Z)[labels[q]])
    paulis.append(op)
    PAULI_LABELS.append(
        " ".join((a + str(q) for q, a in enumerate(labels) if a != "I")) or "I"
    )
PAULIS = np.stack(paulis)
P_REAL = PAULIS.real.copy()
print(f"Observable operator norm: {np.linalg.norm(O, 2):.6f}")


In [ ]:
def encoding_circuit(gamma):
    circuit = Circuit()
    for layer in range(2):
        for qubit in range(NQ):
            angle = f"x{NQ * layer + qubit}"
            rotation = RY if layer == 0 else RZ
            circuit += rotation(angle).on(qubit)
            if gamma:
                noise = (
                    PauliChannel(0, float(gamma), 0)
                    if layer == 0
                    else PauliChannel(0, 0, float(gamma))
                )
                circuit += noise.on(qubit)
        if layer == 0:
            circuit += MQZ.on(1, 0)
            circuit += MQZ.on(2, 1)
            circuit += MQZ.on(3, 2)
    return circuit


def trainable_circuit():
    circuit = Circuit()
    for layer in range(2):
        for qubit in range(NQ):
            circuit += RY(f"theta{NQ * layer + qubit}").on(qubit)
        circuit += MQZ.on(1, 0)
        circuit += MQZ.on(2, 1)
        circuit += MQZ.on(3, 2)
    return circuit


In [ ]:
def readout_coefficients(theta, with_grad=True):
    theta = np.asarray(theta, dtype=np.float64)
    single = theta.ndim == 1
    t = np.atleast_2d(theta)
    rotations = (
        np.cos(t[..., None, None] / 2) * EYE
        + np.sin(t[..., None, None] / 2) * K[np.arange(NPARAM) % NQ]
    )
    l0 = np.broadcast_to(EYE, (len(t), DIM, DIM)).copy()
    l1 = l0.copy()
    for q in range(NQ):
        l0 = rotations[:, q] @ l0
        l1 = rotations[:, NQ + q] @ l1
    v = C @ l1 @ C @ l0
    m = v.transpose(0, 2, 1) @ O @ v
    a = np.einsum("pij,bji->bp", P_REAL, m, optimize=True) / DIM
    if not with_grad:
        return a[0] if single else a
    dv0 = 0.5 * (C @ l1 @ C)[:, None] @ K[None] @ l0[:, None]
    dv1 = 0.5 * C @ K[None] @ l1[:, None] @ C @ l0[:, None]
    dv = np.concatenate([dv0, dv1], axis=1)
    vtdov = v[:, None].transpose(0, 1, 3, 2) @ O @ dv
    dm = vtdov + vtdov.transpose(0, 1, 3, 2)
    da = np.einsum("pij,bqji->bqp", P_REAL, dm, optimize=True) / DIM
    return (a[0], da[0]) if single else (a, da)


def student_unitary(theta):
    v = EYE.copy()
    for layer in range(2):
        for q in range(NQ):
            t = theta[NQ * layer + q]
            v = (np.cos(t / 2) * EYE + np.sin(t / 2) * K[q]) @ v
        v = C @ v
    return v


In [ ]:
axis = 2 * np.pi * np.arange(3) / 3
grid = np.stack(np.meshgrid(*([axis] * NINPUT), indexing="ij"), axis=-1).reshape(
    -1, NINPUT
)

offgrid_reference = np.random.default_rng(20261103).uniform(0, 2 * np.pi, (128, NINPUT))
offgrid_extra = np.random.default_rng(20261104).uniform(0, 2 * np.pi, (1024, NINPUT))
points = np.concatenate([grid, offgrid_reference, offgrid_extra])
FREQUENCIES = np.stack(
    np.meshgrid(*([np.array([0, 1, -1])] * NINPUT), indexing="ij"), axis=-1
).reshape(-1, NINPUT)
ORDERS = abs(FREQUENCIES).sum(axis=1)


In [ ]:
def reference_rho(angles, gamma):
    rho = np.zeros((DIM, DIM), dtype=complex)
    rho[0, 0] = 1
    for layer in range(2):
        for q in range(NQ):
            t = angles[NQ * layer + q]
            if layer == 0:
                u = np.cos(t / 2) * EYE + np.sin(t / 2) * K[q]
                pauli = embed(Y, q)
            else:
                pauli = embed(Z, q)
                u = np.cos(t / 2) * EYE - 1j * np.sin(t / 2) * pauli
            rho = u @ rho @ u.conj().T
            rho = (1 - gamma) * rho + gamma * pauli @ rho @ pauli
        if layer == 0:
            rho = C @ rho @ C
    return rho


def analytic_teacher_terms(points):
    points = np.asarray(points)
    c = np.cos(points)
    s = np.sin(points)
    return (
        np.stack(
            [
                np.sqrt(2) * c[..., 0],
                2 * c[..., 0] * c[..., 1],
                2
                * (
                    c[..., 0] * c[..., 1] * c[..., 2]
                    + s[..., 0] * c[..., 1] * c[..., 4]
                ),
                4 * c[..., 0] * c[..., 1] * c[..., 2] * c[..., 3],
                4
                * np.sqrt(2)
                * s[..., 0]
                * s[..., 1]
                * c[..., 2]
                * s[..., 4]
                * s[..., 5],
                8
                * s[..., 0]
                * c[..., 1]
                * c[..., 2]
                * s[..., 3]
                * c[..., 4]
                * c[..., 7],
                8
                * np.sqrt(2)
                * s[..., 0]
                * s[..., 1]
                * s[..., 2]
                * c[..., 3]
                * s[..., 4]
                * c[..., 5]
                * s[..., 6],
                16
                * s[..., 0]
                * s[..., 1]
                * s[..., 2]
                * s[..., 3]
                * s[..., 4]
                * c[..., 5]
                * c[..., 6]
                * s[..., 7],
            ],
            axis=-1,
        )
        / S
    )


def analytic_teacher(points, gamma=0.0):
    return (
        analytic_teacher_terms(points) * (1 - 2 * gamma) ** np.arange(1, NSECTOR)
    ).sum(axis=-1)


def fft_features(features):
    return (
        np.fft.fftn(
            features.reshape((3,) * NINPUT + (features.shape[-1],)),
            axes=tuple(range(NINPUT)),
        )
        / NGRID
    ).reshape(NGRID, -1)


In [ ]:
states = np.empty((len(GAMMAS), len(points), DIM, DIM), dtype=complex)
density_checks = []
started = time.perf_counter()
for gamma_index, gamma in enumerate(GAMMAS):
    circuit = encoding_circuit(gamma)
    simulator = Simulator("mqmatrix", NQ, dtype=mq.complex128, seed=42)
    for point_index, x in enumerate(points):
        simulator.reset()
        simulator.apply_circuit(
            circuit, {f"x{k}": float(value) for k, value in enumerate(x)}
        )
        states[gamma_index, point_index] = simulator.get_qs()
    rho = states[gamma_index]
    density_checks.append(
        {
            "gamma": float(gamma),
            "max_trace_error": float(abs(np.trace(rho, axis1=-2, axis2=-1) - 1).max()),
            "max_hermiticity_error": float(
                abs(rho - rho.conj().transpose(0, 2, 1)).max()
            ),
            "minimum_eigenvalue": float(np.linalg.eigvalsh(rho).min()),
        }
    )
    print(
        f"Encoded gamma={gamma:.2f}: {len(points)} points ({time.perf_counter()-started:.1f} s elapsed)"
    )


In [ ]:
features_complex = np.einsum("pij,gnji->gnp", PAULIS, states, optimize=True)
assert abs(features_complex.imag).max() < 1e-10
features = features_complex.real
coeff = np.stack([fft_features(r[:NGRID]) for r in features])
a0 = readout_coefficients(np.zeros(NPARAM), False)
targets = features @ a0
print(f"Fourier feature array: {coeff.shape}")


In [ ]:
target_c = coeff @ a0
grams = np.stack([(c.conj().T @ c).real for c in coeff])
roots = np.zeros((len(CONDITIONS), NSECTOR, NPAULI + 1, NPAULI + 1))
cross = []
target_used = []
factor_error = 0.0
for ci, cond in enumerate(CONDITIONS):
    gi = cond["gamma_index"]
    tc = target_c[0 if cond["task"] == "common" else gi]
    target_used.append(tc)
    cross.append((coeff[gi].conj().T @ tc).real)
    for k in range(NSECTOR):
        selected = ORDERS == k
        b = np.column_stack([coeff[gi, selected], -tc[selected]])
        b = np.concatenate([b.real, b.imag], axis=0)
        _, s, vt = np.linalg.svd(b, full_matrices=False)
        d = s[:, None] * vt
        roots[ci, k, : len(s)] = d
        factor_error = max(factor_error, float(abs(b.T @ b - d.T @ d).max()))
cross = np.array(cross)
target_used = np.array(target_used)


In [ ]:
def sector_losses(a, roots):
    z = np.concatenate([a, np.ones(a.shape[:-1] + (1,))], axis=-1)
    residual = np.einsum("bkrp,bp->bkr", roots, z, optimize=True)
    return np.sum(residual * residual, axis=-1)


def loss_gradient(theta, gram, cross):
    a, da = readout_coefficients(theta)
    ga = 2 * (np.einsum("bij,bj->bi", gram, a, optimize=True) - cross)
    grad = np.einsum("bqp,bp->bq", da, ga, optimize=True)
    return (a, grad)


In [ ]:
analytic_target_error = max(
    (
        float(abs(targets[gi] - analytic_teacher(points, float(g))).max())
        for gi, g in enumerate(GAMMAS)
    )
)
_, da0 = readout_coefficients(np.zeros(NPARAM))
first_mask = ORDERS == 1
k1_jacobian_error = 0.0
k1_power_error = 0.0
k1_powers = []
for gi, g in enumerate(GAMMAS):
    actual = coeff[gi] @ da0[0]
    expected_grid = 2 * (1 - 2 * g) / S * np.cos(grid[:, 0])
    expected_coeff = fft_features(expected_grid[:, None]).reshape(-1)
    k1_jacobian_error = max(
        k1_jacobian_error,
        float(abs(actual[first_mask] - expected_coeff[first_mask]).max()),
    )
    power = float(np.sum(abs(actual[first_mask]) ** 2))
    k1_powers.append(power)
    k1_power_error = max(k1_power_error, abs(power - 2 * (1 - 2 * g) ** 2 / S**2))
spectral_error = max(
    (
        float(abs(coeff[gi] - (1 - 2 * g) ** ORDERS[:, None] * coeff[0]).max())
        for gi, g in enumerate(GAMMAS)
    )
)
rng = np.random.default_rng(20261105)
theta = rng.normal(0, 0.5, (7, NPARAM))
a, da = readout_coefficients(theta)
fd_error = 0.0
ps_error = 0.0
for q in range(NPARAM):
    delta = np.zeros(NPARAM)
    delta[q] = 1e-06
    fd = (
        readout_coefficients(theta + delta, False)
        - readout_coefficients(theta - delta, False)
    ) / 2e-06
    fd_error = max(fd_error, float(abs(fd - da[:, q]).max()))
    delta[q] = np.pi / 2
    ps = (
        readout_coefficients(theta + delta, False)
        - readout_coefficients(theta - delta, False)
    ) / 2
    ps_error = max(ps_error, float(abs(ps - da[:, q]).max()))
active_gram = grams[[c["gamma_index"] for c in CONDITIONS]]
losses = sector_losses(a, roots)
_, grad = loss_gradient(theta, active_gram, cross)
fwd_error = 0.0
kraus_error = 0.0
parseval = 0.0
lossgrad_error = 0.0
for ci, cond in enumerate(CONDITIONS):
    gi = cond["gamma_index"]
    f = features[gi, :NGRID] @ a[ci]
    y = targets[0 if cond["task"] == "common" else gi, :NGRID]
    parseval = max(
        parseval, abs(float(np.mean((f - y) ** 2)) - float(losses[ci].sum()))
    )
    for point in [NGRID, NGRID + 1, NGRID + 2]:
        rho = reference_rho(points[point], float(GAMMAS[gi]))
        kraus_error = max(kraus_error, float(abs(rho - states[gi, point]).max()))
        v = student_unitary(theta[ci])
        direct = np.trace(O @ v @ rho @ v.T)
        fwd_error = max(fwd_error, float(abs(direct - features[gi, point] @ a[ci])))
for q in range(NPARAM):
    delta = np.zeros(NPARAM)
    delta[q] = 1e-06
    plus = sector_losses(readout_coefficients(theta + delta, False), roots).sum(axis=1)
    minus = sector_losses(readout_coefficients(theta - delta, False), roots).sum(axis=1)
    lossgrad_error = max(
        lossgrad_error, float(abs((plus - minus) / 2e-06 - grad[:, q]).max())
    )


In [ ]:
gradient_mq_error = 0.0
forward_mq_error = 0.0
qop = QubitOperator()
for label, weight in OBSERVABLE_TERMS:
    qop += QubitOperator(label, float(weight))
hamiltonian = Hamiltonian(qop)
for gamma_index in (0, 2, 3):
    circuit = encoding_circuit(GAMMAS[gamma_index]).as_encoder()
    circuit += trainable_circuit().as_ansatz()
    expectation = Simulator(
        "mqmatrix", NQ, dtype=mq.complex128
    ).get_expectation_with_grad(hamiltonian, circuit)
    parameters = theta[gamma_index]
    readout, readout_gradient = readout_coefficients(parameters)
    values, _, gradients = expectation(points[NGRID : NGRID + 6], parameters)
    forward_mq_error = max(
        forward_mq_error,
        float(
            abs(
                np.asarray(values).reshape(-1)
                - features[gamma_index, NGRID : NGRID + 6] @ readout
            ).max()
        ),
    )
    gradient_mq_error = max(
        gradient_mq_error,
        float(
            abs(
                np.asarray(gradients)[:, 0, :]
                - features[gamma_index, NGRID : NGRID + 6] @ readout_gradient.T
            ).max()
        ),
    )


In [ ]:
teacher_loss = float(sector_losses(np.tile(a0, (7, 1)), roots)[[0, 4, 5, 6]].max())
expected = {k: float(1 / S**2) for k in range(1, NSECTOR)}
target_power = {
    k: float(np.sum(abs(target_c[0, ORDERS == k]) ** 2)) for k in range(NSECTOR)
}
expected_power_error = max((abs(target_power[k] - v) for k, v in expected.items()))
checks = {
    "analytic_teacher_formula": analytic_target_error,
    "first_order_jacobian": k1_jacobian_error,
    "first_order_jacobian_power": k1_power_error,
    "all_pauli_coefficient_identity": spectral_error,
    "factorization_error": factor_error,
    "analytic_vs_finite_difference_coeff_gradient": fd_error,
    "analytic_vs_parameter_shift": ps_error,
    "loss_gradient_finite_difference": lossgrad_error,
    "cached_vs_numpy_kraus": kraus_error,
    "cached_forward_vs_numpy": fwd_error,
    "cached_forward_vs_mindquantum": forward_mq_error,
    "analytic_gradient_vs_mindquantum": gradient_mq_error,
    "parseval": float(parseval),
    "known_teacher_sector_loss": teacher_loss,
    "expected_target_power": expected_power_error,
}
first_step_loss = sector_losses(
    readout_coefficients(theta - 0.1 * grad, False), roots
).sum(axis=1)
initial_loss = losses.sum(axis=1)
one_step_increase = float(np.max(first_step_loss - initial_loss))
jacobian_by_order = []
for gi, gamma in enumerate(GAMMAS):
    for k in range(1, NSECTOR):
        modes = coeff[gi, ORDERS == k]
        dj = np.einsum("mp,bqp->bmq", modes, da, optimize=True)
        powers = np.sum(abs(dj) ** 2, axis=(1, 2))
        jacobian_by_order.append(
            {
                "gamma": float(gamma),
                "order": k,
                "minimum_random_parameter_jacobian_power": float(powers.min()),
            }
        )
good = all(
    (v < 1e-08 if "finite_difference" in k else v < 1e-10 for k, v in checks.items())
)
good = good and all(
    (r["minimum_random_parameter_jacobian_power"] > 1e-16 for r in jacobian_by_order)
)
good = good and min(k1_powers) > 1e-12 and (one_step_increase <= 1e-10)
good = good and float(np.linalg.norm(O, 2)) <= 1 + 1e-10
good = good and all(
    row["max_trace_error"] < 1e-10
    and row["max_hermiticity_error"] < 1e-10
    and row["minimum_eigenvalue"] > -1e-10
    for row in density_checks
)
assert good, "A numerical check failed; inspect the values before training."
print("All numerical checks passed.")


In [ ]:
cache = {"grams": grams, "cross": cross, "sector_roots": roots}


def run_batch(cache, condition_indices, seeds, step_size, steps, label, verbose=True):
    run_count = len(condition_indices) * len(seeds)
    condition_index = np.repeat(condition_indices, len(seeds))
    run_seeds = np.tile(seeds, len(condition_indices))
    initial = {
        seed: np.random.default_rng(int(seed)).normal(0, 0.5, NPARAM) for seed in seeds
    }
    parameters = np.stack([initial[int(seed)] for seed in run_seeds])
    gamma_index = np.array([CONDITIONS[int(i)]["gamma_index"] for i in condition_index])
    active_grams = cache["grams"][gamma_index]
    active_cross = cache["cross"][condition_index]
    active_roots = cache["sector_roots"][condition_index]
    theta_history = np.empty((steps + 1, run_count, NPARAM))
    gradient_history = np.empty_like(theta_history)
    sector_loss_history = np.empty((steps + 1, run_count, NSECTOR))
    started = time.perf_counter()

    for update in range(steps + 1):
        readout, gradient = loss_gradient(parameters, active_grams, active_cross)
        sector_loss = sector_losses(readout, active_roots)
        if not (np.isfinite(sector_loss).all() and np.isfinite(gradient).all()):
            raise FloatingPointError(f"Non-finite values in {label}, update {update}")
        theta_history[update] = parameters
        gradient_history[update] = gradient
        sector_loss_history[update] = sector_loss
        if verbose and (update % 500 == 0 or update == steps):
            print(
                f"{label}: update {update}/{steps}; {time.perf_counter()-started:.1f} s elapsed"
            )
        if update < steps:
            parameters = parameters - step_size * gradient

    return {
        "theta": theta_history,
        "gradient": gradient_history,
        "sector_loss": sector_loss_history,
        "condition_index": condition_index,
        "seeds": run_seeds,
        "eta": np.array(step_size),
        "time": np.arange(steps + 1) * step_size,
        "elapsed_seconds": np.array(time.perf_counter() - started),
    }


In [ ]:
def compare_step_sizes(a, b):
    la = a["sector_loss"]
    lb = b["sector_loss"][::2]
    l0 = la[0].sum(axis=-1)
    mask = la[0] > np.maximum(1e-14, 1e-08 * l0[:, None])
    normdiff = abs(la - lb) / np.maximum(la[0], 1e-300)
    valid = np.broadcast_to(mask, normdiff.shape)
    maximum = float(normdiff[valid].max()) if valid.any() else 0.0
    absdiff = float(
        (abs(la.sum(axis=-1) - lb.sum(axis=-1)) / np.maximum(l0, 1e-14)).max()
    )
    incr = float(np.diff(la.sum(axis=-1), axis=0).max())
    return {
        "all_finite": bool(np.isfinite(la).all() and np.isfinite(lb).all()),
        "max_normalized_sector_difference": maximum,
        "max_total_difference_relative_initial": absdiff,
        "max_single_step_loss_increase": incr,
        "passed": bool(maximum <= 0.05 and absdiff <= 0.05 and (incr <= 1e-10)),
    }


In [ ]:
condition_indices = list(range(len(CONDITIONS)))
step_trials = []
selected_step = None
for candidate in STEP_CANDIDATES:
    pilot_steps = round(PILOT_TIME / candidate)
    pilot = run_batch(
        cache,
        condition_indices,
        PILOT_SEEDS,
        candidate,
        pilot_steps,
        "pilot",
        verbose=False,
    )
    pilot_half = run_batch(
        cache,
        condition_indices,
        PILOT_SEEDS,
        candidate / 2,
        2 * pilot_steps,
        "pilot half-step",
        verbose=False,
    )
    verdict = compare_step_sizes(pilot, pilot_half)
    step_trials.append({"candidate_eta": candidate, **verdict})
    print(
        f"Step {candidate:g}: maximum normalized sector difference = {verdict['max_normalized_sector_difference']:.3e}; passed = {verdict['passed']}"
    )
    if verdict["passed"]:
        selected_step = candidate
        break
assert selected_step is not None, "None of the candidate step sizes passed."
steps = round(TRAINING_TIME / selected_step)
print(f"Selected step size: {selected_step:g}; formal updates: {steps}")


In [ ]:
mainrun = run_batch(cache, condition_indices, SEEDS, selected_step, steps, "Training")
assert mainrun["theta"].shape == (steps + 1, len(CONDITIONS) * len(SEEDS), NPARAM)


In [ ]:
half = run_batch(
    cache,
    condition_indices,
    HALF_STEP_SEEDS,
    selected_step / 2,
    2 * steps,
    "Half-step control",
)
matched_runs = np.flatnonzero(np.isin(mainrun["seeds"], HALF_STEP_SEEDS))
assert np.array_equal(mainrun["condition_index"][matched_runs], half["condition_index"])
assert np.array_equal(mainrun["seeds"][matched_runs], half["seeds"])
slim = {"sector_loss": mainrun["sector_loss"][:, matched_runs]}
half_step_check = compare_step_sizes(slim, half)
print(
    f"Maximum normalized sector-loss difference: {half_step_check['max_normalized_sector_difference']:.6g}"
)
print(
    f"Maximum total-loss difference relative to initialization: {half_step_check['max_total_difference_relative_initial']:.6g}"
)
print(f"Original step-sensitivity criterion passed: {half_step_check['passed']}")


In [ ]:
if SAVE_RESULTS:
    output_dir = OUTPUT_ROOT / datetime.now().strftime("run_%Y%m%d_%H%M%S_%f")
    output_dir.mkdir(parents=True, exist_ok=False)
    np.savez_compressed(output_dir / "training_main.npz", **mainrun)
    np.savez_compressed(output_dir / "training_half_step.npz", **half)
    np.savez_compressed(
        output_dir / "encoder_cache.npz",
        gammas=GAMMAS,
        points=points,
        states=states,
        features=features,
        coefficients=coeff,
        frequencies=FREQUENCIES,
        orders=ORDERS,
        targets=targets,
        target_coefficients=target_used,
        grams=grams,
        cross=cross,
        sector_roots=roots,
        pauli_labels=np.array(PAULI_LABELS),
        a_teacher=a0,
    )
    configuration = {
        "qubits": NQ,
        "input_count": NINPUT,
        "parameter_count": NPARAM,
        "conditions": CONDITIONS,
        "seeds": list(SEEDS),
        "pilot_seeds": list(PILOT_SEEDS),
        "half_step_seeds": list(HALF_STEP_SEEDS),
        "eta": selected_step,
        "steps": steps,
        "optimization_time": TRAINING_TIME,
        "step_trials": step_trials,
        "half_step_check": half_step_check,
        "numpy": np.__version__,
        "mindquantum": mq.__version__,
    }
    (output_dir / "config.json").write_text(json.dumps(configuration, indent=2) + "\n")
    print(f"Saved results to {output_dir}")
else:
    print(
        "Results are in memory. Set SAVE_RESULTS = True and rerun this cell to export them."
    )
